# IEEE 300-bus EMT performance benchmark

This report compares the conventional component model, the inline no-subcomponent conventional variant, individual SSN components, the fused aggregate SSN, and the paper-style grouped passive SSN. The full-duration comparison uses **Δt = 50 μs**, **Tsim = 10 s**, and **200,000 steps**. Generators and operating point are held fixed.

The full-duration rows are single profiled runs, not statistical averages. A separate 1-second solver screen reports its tested repeat count. Internal solver-phase totals may overlap and are diagnostic; they must not be added to step or wall totals. The verified `NoSub` benchmark uses direct PiLine, RXLoad, and Transformer constructors. A midpoint RHS overwrite in the direct Transformer path was corrected; 20,000 IEEE300 steps now match the legacy conventional trace to CSV output precision.

Wall times for SSN, aggregate, and grouped SSN are from the prior recorded wall-time table. Conventional wall times are rounded to 0.1 s from the run-start and profile-output timestamps; accumulated `StepTimes` are the primary comparable measurement.

The IEEE300 executable defaults to the fastest measured formulation: fused aggregate SSN with conventional VBR generators and standard KLU numeric refactorization (partial refactorization explicitly disabled).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("examples/Notebooks/Benchmarks/data/IEEE300_Publishable_10s_50us")
if not DATA.exists():
    DATA = Path("data/IEEE300_Publishable_10s_50us")
summary = pd.read_csv(DATA / "benchmark_summary.csv")
screen = pd.read_csv(DATA / "solver_screen_summary.csv")
traces = pd.read_csv(DATA / "trace_validation.csv")
summary["step_ms_per_step"] = summary.step_total_s / summary.steps * 1e3
summary[
    [
        "case",
        "formulation",
        "solver",
        "steps",
        "step_total_s",
        "wall_total_s",
        "step_ms_per_step",
    ]
]

matrix_stems = {
    "Conventional": "conventional_legacy",
    "Conventional_NoSub": "conventional_NoSub_AllDirect",
    "SSN": "ssn",
    "Aggregate": "aggregate",
    "Grouped passive SSN": "grouped_default",
}
matrix_rows = []
for label, stem in matrix_stems.items():
    m = pd.read_csv(DATA / f"{stem}_SolverMatrixStats.csv").iloc[0]
    matrix_rows.append(
        {
            "configuration": label,
            "rows": int(m.rows),
            "columns": int(m.cols),
            "nonzeros": int(m.nonzeros),
            "variable_entries": int(m.variable_entries),
        }
    )
matrix_summary = pd.DataFrame(matrix_rows)
matrix_summary

In [ ]:
# Primary comparison: cumulative measured step time and end-to-end wall time.
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), constrained_layout=True)
colors = ["#777777", "#59A14F", "#4E79A7", "#F28E2B", "#B07AA1", "#E15759", "#76B7B2"]
x = np.arange(len(summary))
axes[0].bar(x, summary.step_total_s, color=colors[: len(x)])
axes[0].set_title("Accumulated step time")
axes[0].set_ylabel("seconds over 200,000 steps")
axes[0].set_xticks(x, summary.case, rotation=25, ha="right")
axes[0].grid(axis="y", alpha=0.25)
wall = summary.wall_total_s.astype(float)
axes[1].bar(x, wall, color=colors[: len(x)])
axes[1].set_title("Measured wall time")
axes[1].set_ylabel("seconds")
axes[1].set_xticks(x, summary.case, rotation=25, ha="right")
axes[1].grid(axis="y", alpha=0.25)
for ax, values in zip(axes, [summary.step_total_s, wall]):
    for i, v in enumerate(values):
        if pd.notna(v):
            ax.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)
fig.suptitle("IEEE300 EMT | Δt = 50 μs | Tsim = 10 s | one full run per configuration")
plt.show()

In [ ]:
# Solver internals: compare numeric solve, matrix/refactor, and RHS assembly.
# The profile field named partial_refactorization includes KLU's ordinary full
# numeric refactor call when partial refactorization is disabled. These timers
# are nested diagnostic timers and overlap; do not sum them into wall time.
configs = {
    "Conventional": "conventional_legacy",
    "Conventional_NoSub": "conventional_NoSub_AllDirect",
    "SSN": "ssn",
    "Aggregate": "aggregate",
    "Grouped DenseLU static": "grouped_default",
    "Grouped DenseLU exact": "grouped_denseexact_10s",
}
wanted = [
    "numeric_solve",
    "partial_refactorization",
    "rhs_assembly",
    "matrix_reset_and_variable_stamping",
]
phase_rows = []
for label, stem in configs.items():
    frame = pd.read_csv(DATA / f"{stem}_SolverPhaseTimes.csv").set_index("phase")
    for phase in wanted:
        phase_rows.append(
            {
                "configuration": label,
                "phase": phase,
                "seconds": frame.loc[phase, "total_s"],
            }
        )
phase = (
    pd.DataFrame(phase_rows)
    .pivot(index="configuration", columns="phase", values="seconds")
    .reindex(configs)
)
ax = phase[wanted].plot.bar(
    figsize=(13, 5.5), width=0.8, color=["#4E79A7", "#F28E2B", "#59A14F", "#B07AA1"]
)
ax.set_title("Nested MNA/solver phase totals | IEEE300 EMT | 10 s at 50 μs")
ax.set_ylabel("seconds")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()
phase

In [ ]:
# Solver/backend comparison. The 1-second screen is separated from the full 10-second results.
screen[
    [
        "formulation",
        "solver",
        "sim_duration_s",
        "steps",
        "step_total_s",
        "replicates",
        "status",
        "notes",
    ]
]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
ok = screen[screen.step_total_s.notna()].copy()
labels = [
    f"{r.solver}\n{r.formulation}\n{r.sim_duration_s:g} s, n={r.replicates}"
    for r in ok.itertuples()
]
bars = ax.bar(
    np.arange(len(ok)),
    ok.step_total_s,
    color=["#F28E2B" if "DenseLU exact" in s else "#4E79A7" for s in ok.solver],
)
ax.set_xticks(np.arange(len(ok)), labels, rotation=15, ha="right")
ax.set_ylabel("accumulated step time (s)")
ax.set_title("Solver screen and full-run checkpoints — durations are shown per bar")
ax.grid(axis="y", alpha=0.25)
for b, v in zip(bars, ok.step_total_s):
    ax.text(
        b.get_x() + b.get_width() / 2,
        v,
        f"{v:.1f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )
plt.show()

exact_repeat = screen[
    (screen.formulation == "Grouped passive SSN")
    & (screen.solver == "DenseLU exact solve")
    & (screen.sim_duration_s == 1)
].iloc[0]
print(
    f"DenseLU exact, grouped SSN, 1 s screen: mean={exact_repeat.step_total_s:.3f} s, {exact_repeat.notes}"
)

## Trace validation and interpretation

The verified `conventional nosub` alias was compared against the legacy IEEE300 conventional topology for 20,000 steps at 50 μs. All 240,024 logged voltage values match to six-decimal CSV output precision; the largest displayed difference is 1 μV.

The direct Transformer trace issue was traced to two inductor-history contributions sharing the midpoint RHS row. After combining that row correctly, its isolated IEEE118 comparison also passes (0.000232 V maximum difference).

The verified `conventional nosub` alias uses direct inline PiLine, RXLoad, and Transformer updates. It removes child-component/scheduler overhead while keeping the global MNA matrix at 2,907 × 2,907; this is not a matrix-size reduction.

## Interpretation

- The fused aggregate SSN is the fastest measured 10-second configuration (58.09 s accumulated step time; 59.55 s wall time) and is the IEEE300 default. It uses KLU's ordinary full numeric refactorization on each variable-matrix update; partial refactorization is explicitly disabled.
- The all-direct conventional `NoSub` path took 59.78 s accumulated step time (about 61.0 s wall) versus 146.64 s legacy conventional. It is the fastest conventional formulation measured here, but the fused aggregate remains slightly faster at 58.09 s.
- Grouped passive SSN is dominated by reduced-history/RHS work; exact DenseLU removes the iterative correction/fallback cost, yet the 10-second grouped run is still much slower than aggregate SSN.
- SparseLU did not complete: it threw `std::bad_alloc` while initializing the grouped system. KLU partial refactorization was tested on the same 1-second grouped screen and was slower than the five-run DenseLU-exact mean.

## Reproduction

From the repository root, use the built IEEE300 executable. The first command is the default fastest configuration; the second uses direct constructors for PiLine, RXLoad, and Transformer; the third retains legacy subcomponents for comparison. `profile` disables the data logger and writes detailed timing files.

```sh
build/dpsim/examples/cxx/IEEE300_Ph3 emt profile dt50us aggregate 10s
build/dpsim/examples/cxx/IEEE300_Ph3 emt conventional nosub profile dt50us 10s
build/dpsim/examples/cxx/IEEE300_Ph3 emt conventional profile dt50us 10s
build/dpsim/examples/cxx/IEEE300_Ph3 emt allssn paper denseexact groupshared profile dt50us 10s
```

All times are machine- and build-dependent; this notebook records the exact measured run files and should be rerun on the target machine before publication or external claims.